In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet("./batch_0.parquet")

In [ ]:
df["msa"].dropna(inplace=True)

In [ ]:
df["msa"].nunique()

In [ ]:
df.head()

In [ ]:
import plotly.express as px

# Map full state names to USPS 2-letter abbreviations (Plotly USA-states needs abbreviations)
state_to_abbrev = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC"
}

# Count entries by state
state_counts = (
    df.assign(state_clean=df["state"].astype("string").str.strip())
      .dropna(subset=["state_clean"])
      .assign(state_abbrev=lambda x: x["state_clean"].map(state_to_abbrev))
      .dropna(subset=["state_abbrev"])
      .groupby("state_abbrev", as_index=False)
      .size()
      .rename(columns={"size": "entry_count"})
      .sort_values("entry_count", ascending=False)
)

# Quick quality check for state names that did not map
unknown_states = sorted(set(df["state"].dropna().astype(str).str.strip()) - set(state_to_abbrev.keys()))
print(f"Mapped states: {state_counts.shape[0]}")
print(f"Unknown/unmapped state labels: {len(unknown_states)}")
if unknown_states:
    print(unknown_states[:20])

fig = px.choropleth(
    state_counts,
    locations="state_abbrev",
    locationmode="USA-states",
    color="entry_count",
    scope="usa",
    color_continuous_scale="Blues",
    labels={"entry_count": "# entries"},
    title="Entries per US State"
)

fig.update_traces(marker_line_color="white", marker_line_width=0.8)
fig.update_layout(margin=dict(l=10, r=10, t=50, b=10))
fig.show()

In [ ]:
# --- MSA boundary choropleth ---
import re
from pathlib import Path
from urllib.request import urlretrieve

import plotly.express as px
import json

try:
    import geopandas as gpd
except ImportError as e:
    raise ImportError("geopandas is required for MSA boundary mapping. Install with: pip install geopandas") from e

CBSA_URL = "https://www2.census.gov/geo/tiger/TIGER2023/CBSA/tl_2023_us_cbsa.zip"
data_dir = Path("./data")
data_dir.mkdir(exist_ok=True)
cbsa_zip = data_dir / "tl_2023_us_cbsa.zip"

if not cbsa_zip.exists():
    print(f"Downloading CBSA boundaries from Census: {CBSA_URL}")
    urlretrieve(CBSA_URL, cbsa_zip)

cbsa_gdf = gpd.read_file(f"zip://{cbsa_zip.resolve()}")
name_col = "NAMELSAD" if "NAMELSAD" in cbsa_gdf.columns else "NAME"

def normalize_msa_name(text):
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return None
    s = str(text).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = s.replace(" metropolitan statistical area", "")
    s = s.replace(" micropolitan statistical area", "")
    s = re.sub(r"\bmetro area\b", "", s)
    s = re.sub(r"\bmicro area\b", "", s)
    s = re.sub(r"\bmsa\b", "", s)
    s = s.replace("-", " ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Build counts from df['msa']
msa_counts = (
    df[["msa"]]
      .dropna(subset=["msa"])
      .assign(msa_raw=lambda x: x["msa"].astype(str).str.strip())
      .assign(state_abbrev=lambda x: x["msa_raw"].str.extract(r"\b([A-Z]{2})\s+MSA$"))
      .assign(msa_name_only=lambda x: x["msa_raw"].str.replace(r"\s+[A-Z]{2}\s+MSA$", "", regex=True))
      .assign(join_name=lambda x: x["msa_name_only"].map(normalize_msa_name))
      .groupby(["join_name", "state_abbrev"], as_index=False)
      .size()
      .rename(columns={"size": "entry_count"})
      .sort_values("entry_count", ascending=False)
)

# Prepare CBSA boundaries for join
cbsa_gdf = cbsa_gdf.assign(
    state_abbrev=cbsa_gdf[name_col].astype(str).str.extract(r",\s*([A-Z]{2})\b"),
    msa_name_only=cbsa_gdf[name_col].astype(str).str.replace(r",\s*[A-Z]{2}.*$", "", regex=True),
)
cbsa_gdf["join_name"] = cbsa_gdf["msa_name_only"].map(normalize_msa_name)

msa_map_gdf = cbsa_gdf.merge(msa_counts, on=["join_name", "state_abbrev"], how="left")
msa_map_gdf["entry_count"] = msa_map_gdf["entry_count"].fillna(0)

# Match quality review
matched_rows = int((msa_map_gdf["entry_count"] > 0).sum())
print(f"CBSA polygons with data: {matched_rows:,}")
print(f"Total distinct MSA keys in data: {msa_counts.shape[0]:,}")

matched_keys = set(
    msa_map_gdf.loc[msa_map_gdf["entry_count"] > 0, ["join_name", "state_abbrev"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

unmatched = msa_counts[
    ~msa_counts[["join_name", "state_abbrev"]]
      .apply(tuple, axis=1)
      .isin(matched_keys)
]

if not unmatched.empty:
    print("Top unmatched MSA labels (review):")
    display(unmatched.head(20))

# Plotly interactive choropleth
msa_map_gdf = msa_map_gdf.to_crs(epsg=4326)
plot_df = msa_map_gdf[["GEOID", name_col, "entry_count", "geometry"]].copy()
plot_df["entry_count"] = plot_df["entry_count"].fillna(0)
plot_df["entry_count"] = plot_df["entry_count"].astype(float)

geojson = json.loads(plot_df.to_json())

fig = px.choropleth(
    plot_df,
    geojson=geojson,
    locations="GEOID",
    featureidkey="properties.GEOID",
    color="entry_count",
    hover_name=name_col,
    hover_data={"entry_count": ":,.0f", "GEOID": True},
    color_continuous_scale="Blues",
    labels={"entry_count": "# entries"},
    title="Entries per MSA (Interactive Census CBSA Choropleth)",
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_traces(marker_line_color="white", marker_line_width=0.3)
fig.update_layout(margin=dict(l=10, r=10, t=50, b=10), coloraxis_colorbar_title="# entries")
fig.show()

# Generate mapping MSA to State

In [ ]:
from pathlib import Path
import geopandas as gpd

import requests
from pathlib import Path
from tqdm.auto import tqdm

# Ensure data directory exists
data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)

county_zip = data_dir / "tl_2023_us_county.zip"
url = "https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip"

if not county_zip.exists():
    print(f"Downloading {url}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    
    with open(county_zip, 'wb') as file, tqdm(
        desc=county_zip.name,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)
    print("Download complete! You can now run the geopandas mapping cell.")
else:
    print("File already exists! You are good to go.")
    
# Switch to the county shapefile which contains both FIPS codes
county_zip = Path("./data/tl_2023_us_county.zip")
if not county_zip.exists():
    raise FileNotFoundError("Missing ./data/tl_2023_us_county.zip. Please download the county TIGER/Line shapefile.")

county_gdf = gpd.read_file(f"zip://{county_zip.resolve()}")

state_cbsa_fips_map = (
    county_gdf[["STATEFP", "CBSAFP"]]
      .rename(columns={"STATEFP": "state_fips", "CBSAFP": "cbsa_fips"})
      # Drop counties that don't belong to any CBSA
      .dropna(subset=["state_fips", "cbsa_fips"])
      .astype({"state_fips": "string", "cbsa_fips": "string"})
      # Some shapefile readers load empty strings instead of nulls, so filter them out
      .query("cbsa_fips != ''")
      .drop_duplicates()
      .sort_values(["state_fips", "cbsa_fips"])
      .reset_index(drop=True)
)

print(f"Unique state_fips/cbsa_fips pairs: {state_cbsa_fips_map.shape[0]:,}")
display(state_cbsa_fips_map.head(20))

# Optional export
state_cbsa_fips_map.to_parquet("./data/state_cbsa_fips_map.parquet", index=False)
state_cbsa_fips_map.to_csv("./data/state_cbsa_fips_map.csv", index=False)
print("Saved: ./data/state_cbsa_fips_map.parquet and ./data/state_cbsa_fips_map.csv")

# Generate state geojson

In [ ]:
from pathlib import Path
import geopandas as gpd
import requests
from tqdm.auto import tqdm

data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)

state_zip = data_dir / "tl_2023_us_state.zip"
state_url = "https://www2.census.gov/geo/tiger/TIGER2023/STATE/tl_2023_us_state.zip"

if not state_zip.exists():
    print(f"Downloading {state_url}...")
    response = requests.get(state_url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get("content-length", 0))
    with open(state_zip, "wb") as file, tqdm(
        desc=state_zip.name,
        total=total_size,
        unit="iB",
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

states_gdf = gpd.read_file(f"zip://{state_zip.resolve()}")

states_gdf["state_code"] = states_gdf["STATEFP"].astype(str).str.zfill(2)

states_out = states_gdf[["state_code", "NAME", "geometry"]].copy()
states_out.to_file("./data/us_states.geojson", driver="GeoJSON")

print("Saved ./data/us_states.geojson")

# Generate CBSA

In [ ]:
cbsa_zip = data_dir / "tl_2023_us_cbsa.zip"
cbsa_url = "https://www2.census.gov/geo/tiger/TIGER2023/CBSA/tl_2023_us_cbsa.zip"

if not cbsa_zip.exists():
    print(f"Downloading {cbsa_url}...")
    response = requests.get(cbsa_url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get("content-length", 0))
    with open(cbsa_zip, "wb") as file, tqdm(
        desc=cbsa_zip.name,
        total=total_size,
        unit="iB",
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

cbsa_gdf = gpd.read_file(f"zip://{cbsa_zip.resolve()}")

cbsa_gdf["msa_code"] = cbsa_gdf["CBSAFP"].astype(str).str.zfill(5)

#filter to keep metropolitan areas only (LSAD == "M1")
cbsa_gdf = cbsa_gdf[cbsa_gdf["LSAD"] == "M2"].copy()
# Rename for frontend clarity
cbsa_gdf = cbsa_gdf.rename(columns={"NAMELSAD": "name"})

# Drop duplicates / bad rows
cbsa_gdf = cbsa_gdf.dropna(subset=["msa_code", "geometry"])


cbsa_out = cbsa_gdf[["msa_code", "name", "geometry"]].copy()
cbsa_out.to_file("./data/us_msas.geojson", driver="GeoJSON")

print("Saved ./data/us_msas.geojson")

In [ ]:
msa_map_gdf[msa_map_gdf["CBSAFP"].isin(cbsa_out["msa_code"])].shape
                    